# Company Quarter Prompt Search

In [1]:
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

from src.data_collection.consts import DB_PARAMS
from src.data_collection.model_driver.model_driver_class import ModelDriver
from src.data_analysis.data_fetcher.data_fetcher_class import DataFetcher

/home/maxim-shibanov/anaconda3/envs/vllm_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
fq_verbolizer = {
    1: ["first", "january", "february", "march", "spring", "earlyyear"],
    2: ["second", "april", "may", "june", "midyear"],
    3: ["third", "july", "august", "september", "lateyear"],
    4: ["fourth", "october", "november", "december", "yearend", "annual", "yearly"],
}

quarter_prompts = [
    "Read the report excerpt and identify the calendar quarter of the SEC filing date. Selected quarter:",
    "Classify the calendar quarter of the filing date for this SEC report. Classification:",
    "You are tagging SEC filings by submission quarter. From this report excerpt, the filing belongs to the",
    "Imagine you are reviewing SEC metadata and need to assign the filing to a calendar quarter. The correct quarter is",
    "Treat this as a metadata classification task for an SEC document. Your goal is not to identify the reporting period quarter, but the actual calendar quarter in which the filing was submitted to the SEC. Based on the excerpt, the filing quarter should be labeled:",
    "Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:", # The best 
    "Classify the company quarter rank for this SEC report within the company year. Classification:",
    "You are tagging company reports by quarter rank within the company year. From this report excerpt, the company quarter is",
    "Imagine you are reviewing company reporting metadata and need to assign the report to the correct company quarter within the year. The correct company quarter is",
    "Treat this as a metadata classification task for an SEC document. Your goal is to identify the company's quarter rank within its reporting year. Based on the excerpt, the company quarter should be labeled:",
    "Using the excerpt, infer where this report falls in the company's own yearly reporting sequence. Choose among first, second, third, and fourth. The company quarter rank is:",
    "Ignore the SEC submission month and focus on the company's reporting cycle. Which company quarter does this report represent within the year? Answer:",
    "Determine the quarter number of this report within the company's annual reporting pattern, rather than the calendar filing quarter. The correct company quarter is:",
    "Based on the reporting language and report structure, assign this filing to the company's first, second, third, or fourth reporting quarter of the year. Company quarter:",
    "You are labeling reports by quarter order within each company's reporting year. From this excerpt, the most likely company quarter rank is:",
]
quarter_prompts = [
    "Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:",
    "Ignore the SEC submission month and focus on the company's reporting cycle. Which company quarter does this report represent within the year? Answer:",
]

In [3]:
model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "12GiB", "cpu": "64GiB"},
)

model_driver = ModelDriver(model_name=model_name, model=model)

Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.12it/s]
2026-06-16 16:00:16,633 [WARNING] Some parameters are on the meta device because they were offloaded to the cpu.


In [16]:
def build_balanced_quarter_sample(
    data: pd.DataFrame,
    label_col: str,
    sample_size_per_quarter: int = 10,
    random_state: int = 427837832,
) -> pd.DataFrame:
    quarter_samples = []
    for quarter_label in [1, 2, 3, 4]:
        quarter_df = data[data[label_col] == quarter_label].copy()
        if len(quarter_df) < sample_size_per_quarter:
            raise ValueError(
                f"Not enough rows for quarter {quarter_label} in '{label_col}': "
                f"expected at least {sample_size_per_quarter}, got {len(quarter_df)}."
            )
        quarter_samples.append(
            quarter_df.sample(n=sample_size_per_quarter, random_state=random_state)
        )

    sample_df = pd.concat(quarter_samples, ignore_index=True)
    sample_df = sample_df.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return sample_df


fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=["raw_text"])
reports_df = reports_df[reports_df["report_type"].isin(["10-K", "10-Q"])].copy()
reports_df["filed_date"] = pd.to_datetime(reports_df["filed_date"])
reports_df = reports_df.sort_values("filed_date").copy()

quarter_labels_df = fetcher.derive_company_quarter_labels_by_cycle(
    reports_df,
    report_id_col="id",
    output_col="true_label",
)
reports_df = reports_df.merge(quarter_labels_df, on="id", how="inner")
reports_df["true_label"] = reports_df["true_label"].astype(int)

sample_df = build_balanced_quarter_sample(reports_df, label_col="true_label")
display(sample_df[["id", "report_type", "filed_date", "true_label"]].head())


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:230: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:211: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:249: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)


,id,report_type,filed_date,true_label
0,21424,10-Q,2019-01-30,1
1,29966,10-Q,2019-07-24,2
2,102033,10-Q,2022-10-28,3
3,72806,10-Q,2021-10-28,3
4,20686,10-K,2019-02-21,4


In [17]:
sample_df["id"]
len(sample_df["id"])

40

In [6]:
def evaluate_prompt(prompt: str, data: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    rows = []
    for row in tqdm(data.itertuples(index=False), total=len(data), desc=prompt[:48]):
        result = model_driver.predict_metadata(
            verbolizer=fq_verbolizer,
            prompt=prompt,
            text=row.raw_text,
        )
        probabilities = result["probabilities"]
        predicted_label = max(probabilities, key=probabilities.get)
        rows.append(
            {
                "id": row.id,
                "report_type": row.report_type,
                "filed_date": row.filed_date,
                "true_label": row.true_label,
                "predicted_label": predicted_label,
                "confidence": result["confidence"],
                "true_label_probability": probabilities[row.true_label],
            }
        )

    predictions_df = pd.DataFrame(rows)
    y_true = predictions_df["true_label"]
    y_pred = predictions_df["predicted_label"]

    metrics = {
        "prompt": prompt,
        "avg_confidence": predictions_df["confidence"].mean(),
        "avg_true_label_probability": predictions_df["true_label_probability"].mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    return metrics, predictions_df


all_metrics = []
all_predictions = {}

for prompt in quarter_prompts:
    metrics, predictions_df = evaluate_prompt(prompt, sample_df)
    all_metrics.append(metrics)
    all_predictions[prompt] = predictions_df

    print("\nPROMPT:", prompt)
    print(predictions_df["predicted_label"].value_counts())
    print(pd.crosstab(predictions_df["true_label"], predictions_df["predicted_label"]))

results_df = pd.DataFrame(all_metrics)
results_df = results_df.sort_values(["f1_macro", "avg_confidence"], ascending=[False, False]).reset_index(drop=True)
display(results_df)


Read the report excerpt and identify this compan: 100%|██████████| 40/40 [01:41<00:00,  2.55s/it]



PROMPT: Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:
predicted_label
4    13
1    11
2     8
3     8
Name: count, dtype: int64
predicted_label  1  2  3   4
true_label                  
1                7  0  1   2
2                2  7  0   1
3                2  1  7   0
4                0  0  0  10


Ignore the SEC submission month and focus on the: 100%|██████████| 40/40 [01:43<00:00,  2.58s/it]


PROMPT: Ignore the SEC submission month and focus on the company's reporting cycle. Which company quarter does this report represent within the year? Answer:
predicted_label
4    17
3    10
1     8
2     5
Name: count, dtype: int64
predicted_label  1  2  3   4
true_label                  
1                6  0  1   3
2                1  4  2   3
3                1  1  7   1
4                0  0  0  10


,prompt,avg_confidence,avg_true_label_probability,accuracy,precision_macro,recall_macro,f1_macro
0,Read the report excerpt and identify this comp...,0.069477,0.447149,0.775,0.788899,0.775,0.772947
1,Ignore the SEC submission month and focus on t...,0.114033,0.414809,0.675,0.709559,0.675,0.660185


In [7]:
results_df = results_df.sort_values(["f1_macro", "avg_confidence"], ascending=[False, False]).reset_index(drop=True)
display(results_df)


,prompt,avg_confidence,avg_true_label_probability,accuracy,precision_macro,recall_macro,f1_macro
0,Read the report excerpt and identify this comp...,0.069477,0.447149,0.775,0.788899,0.775,0.772947
1,Ignore the SEC submission month and focus on t...,0.114033,0.414809,0.675,0.709559,0.675,0.660185


In [8]:
top_3_prompts = results_df.head(3).copy()
display(top_3_prompts[["prompt", "avg_confidence", "avg_true_label_probability", "precision_macro", "recall_macro", "f1_macro", "accuracy"]])

for idx, row in top_3_prompts.iterrows():
    print(f"\nTop {idx + 1} prompt")
    print(row["prompt"])
    print(
        {
            "avg_confidence": row["avg_confidence"],
            "avg_true_label_probability": row["avg_true_label_probability"],
            "precision_macro": row["precision_macro"],
            "recall_macro": row["recall_macro"],
            "f1_macro": row["f1_macro"],
            "accuracy": row["accuracy"],
        }
    )


,prompt,avg_confidence,avg_true_label_probability,precision_macro,recall_macro,f1_macro,accuracy
0,Read the report excerpt and identify this comp...,0.069477,0.447149,0.788899,0.775,0.772947,0.775
1,Ignore the SEC submission month and focus on t...,0.114033,0.414809,0.709559,0.675,0.660185,0.675



Top 1 prompt
Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:
{'avg_confidence': 0.06947727333754301, 'avg_true_label_probability': 0.4471485440591742, 'precision_macro': 0.7888986013986014, 'recall_macro': 0.7749999999999999, 'f1_macro': 0.7729468599033816, 'accuracy': 0.775}

Top 2 prompt
Ignore the SEC submission month and focus on the company's reporting cycle. Which company quarter does this report represent within the year? Answer:
{'avg_confidence': 0.1140330605674535, 'avg_true_label_probability': 0.4148090220982799, 'precision_macro': 0.7095588235294118, 'recall_macro': 0.675, 'f1_macro': 0.6601851851851852, 'accuracy': 0.675}


In [9]:
# Optional: inspect one prediction with top-token printing.
example_prompt = top_3_prompts.iloc[0]["prompt"]
example_report = sample_df.iloc[0]["raw_text"]
model_driver.predict_metadata(
    verbolizer=fq_verbolizer,
    prompt=example_prompt,
    text=example_report,
    print_top_tokens=True,
    top_k_tokens=20,
)


Top 20 tokens by probability:
HO           | 0.074219
Quarter      | 0.027344
Table        | 0.015564
Select       | 0.014648
Click        | 0.013306
To           | 0.008057
Company      | 0.007599
Qu           | 0.007141
View         | 0.006287
Ex           | 0.005219
End          | 0.005219
December     | 0.005066
quarter      | 0.004608
Ho           | 0.004456
Hol          | 0.004333
Choose       | 0.004333
Financial    | 0.004181
The          | 0.004059
SE           | 0.003815
Report       | 0.003693


{'confidence': 0.026086539030075073,
 'probabilities': {4: 0.373811147707293,
  1: 0.2902347601274507,
  3: 0.2681993650314914,
  2: 0.06775472713376494}}

In [10]:


quarter_prompt = "Read the report excerpt and identify this company's quarter within its reporting year. Selected company quarter:"
fq_verbolizer = {
    1: ["first", "january", "february", "march", "spring", "earlyyear"],
    2: ["second", "april", "may", "june", "midyear"],
    3: ["third", "july", "august", "september", "lateyear"],
    4: ["fourth", "october", "november", "december", "yearend", "annual", "yearly"],
}

ids = [88, 713, 2179, 2368, 2417, 2560, 2678, 3058, 3757, 3990]

fetcher = DataFetcher(DB_PARAMS)
reports_df = fetcher._fetch_reports(regressors=["raw_text"])
reports_df = reports_df[reports_df["report_type"].isin(["10-K", "10-Q"])].copy()

quarter_labels_df = fetcher.derive_company_quarter_labels_by_cycle(
    reports_df,
    report_id_col="id",
    output_col="true_label",
)

reports_df = reports_df.merge(quarter_labels_df, on="id", how="inner")
sample_df = reports_df[reports_df["id"].isin(ids)].copy().sort_values("id")

model_name = "mistralai/Mistral-7B-v0.1"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    max_memory={0: "12GiB", "cpu": "64GiB"},
)
model_driver = ModelDriver(model_name=model_name, model=model)

rows = []
for row in sample_df.itertuples(index=False):
    result = model_driver.predict_metadata(
        verbolizer=fq_verbolizer,
        prompt=quarter_prompt,
        text=row.raw_text,
    )
    probs = result["probabilities"]
    rows.append({
        "id": row.id,
        "true_label": row.true_label,
        "predicted_label": max(probs, key=probs.get),
        "confidence": result["confidence"],
        "probabilities": probs,
    })

pred_df = pd.DataFrame(rows)
display(pred_df[["id", "true_label", "predicted_label", "confidence"]])
display(pd.crosstab(pred_df["true_label"], pred_df["predicted_label"]))
pred_df

Available regressors:
 - avg_default_verbolizer
 - avg_shrink_verbolizer
 - doc_len
 - eps_surprise
 - f_size
 - full_list_default_verbolizer
 - full_list_shrink_verbolizer
 - hv_orig_score
 - lm_orig_score
 - max_abs_default
 - max_abs_shrink
 - max_default_verbolizer
 - max_shrink_verbolizer
 - md_hv1
 - md_hv2
 - md_hv3
 - md_lm1
 - md_lm2
 - md_lm3
 - min_default_verbolizer
 - min_shrink_verbolizer
 - stretch_default
 - stretch_shrink
Available sectors:
 - Technology (92)
 - Industrials (86)
 - Financial Services (85)
 - Healthcare (66)
 - Consumer Cyclical (58)
 - Consumer Defensive (40)
 - Real Estate (32)
 - Utilities (32)
 - Energy (30)
 - Basic Materials (23)
 - Communication Services (22)


/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:230: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:211: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)
/home/maxim-shibanov/Projects_Py/Risk-and-return-prediction-with-LLM/src/data_analysis/data_fetcher/data_fetcher_class.py:249: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacity of 15.55 GiB of which 122.56 MiB is free. Including non-PyTorch memory, this process has 14.35 GiB memory in use. Of the allocated memory 13.61 GiB is allocated by PyTorch, and 459.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)